# 07 Calibration Evaluation

This notebook is the main numerical experiment results notebook.

It evaluates probabilistic VAR forecasts generated in `06_forecasts.ipynb` across:

- four controlled DGPs,
- four innovation models,
- multiple calibration and scoring metrics.

The central question is:

**When do flexible innovation models materially improve probabilistic forecast calibration under innovation misspecification?**

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.append(str(ROOT))

import numpy as np
import pandas as pd

from src.utils.seeds import set_seed

from src.experiments.config import ExperimentConfig
from src.experiments.paths import result_dirs
from src.experiments.artifacts import (
    save_table,
    save_json,
    save_current_figure,
)

from src.api.forecasts import load_forecast_store

set_seed(123)

config = ExperimentConfig()

dgp_names = config.dgp_names

forecast_models = [
    "VAR",
    "RNN",
]

innovation_model_names = [
    "gaussian",
    "bootstrap",
    "student_t",
    "diffusion",
]

dirs = result_dirs(
    "07_calibration_evaluation",
    test=False,
)

forecast_dir = (
    ROOT
    / "results"
    / "forecasts"
    / "06_forecasts"
)

interval = (0.05, 0.95)
nominal_levels = config.nominal_levels

forecast_store = load_forecast_store(
    forecast_dir=forecast_dir,
    forecast_models=forecast_models,
    dgp_names=dgp_names,
    innovation_model_names=innovation_model_names,
)

VAR gaussian gaussian (250, 40, 3) (40, 3)
VAR gaussian bootstrap (250, 40, 3) (40, 3)
VAR gaussian student_t (250, 40, 3) (40, 3)
VAR gaussian diffusion (250, 40, 3) (40, 3)
VAR student_t gaussian (250, 40, 3) (40, 3)
VAR student_t bootstrap (250, 40, 3) (40, 3)
VAR student_t student_t (250, 40, 3) (40, 3)
VAR student_t diffusion (250, 40, 3) (40, 3)
VAR mixture gaussian (250, 40, 3) (40, 3)
VAR mixture bootstrap (250, 40, 3) (40, 3)
VAR mixture student_t (250, 40, 3) (40, 3)
VAR mixture diffusion (250, 40, 3) (40, 3)
VAR heteroskedastic gaussian (250, 40, 3) (40, 3)
VAR heteroskedastic bootstrap (250, 40, 3) (40, 3)
VAR heteroskedastic student_t (250, 40, 3) (40, 3)
VAR heteroskedastic diffusion (250, 40, 3) (40, 3)
RNN gaussian gaussian (250, 40, 3) (40, 3)
RNN gaussian bootstrap (250, 40, 3) (40, 3)
RNN gaussian student_t (250, 40, 3) (40, 3)
RNN gaussian diffusion (250, 40, 3) (40, 3)
RNN student_t gaussian (250, 40, 3) (40, 3)
RNN student_t bootstrap (250, 40, 3) (40, 3)
RNN stud

In [2]:
n_loaded = sum(
    len(forecast_store[forecast_model][dgp_name])
    for forecast_model in forecast_models
    for dgp_name in dgp_names
)

print("Loaded forecast objects:", n_loaded)

Loaded forecast objects: 32


## Main Forecast Evaluation Table

The first evaluation step computes a unified set of probabilistic forecast metrics for each DGP and innovation model.

Metrics include:

- empirical interval coverage,
- average interval width,
- expected calibration error,
- PIT deviation from uniformity,
- CRPS,
- energy score,
- interval score.

Lower is better for ECE, PIT deviation, CRPS, energy score, and interval score. Coverage should be close to the nominal level.

In [3]:
from src.api.calibration import calibration_results_table

main_results_df, summary_objects = calibration_results_table(
    forecast_store=forecast_store,
    forecast_models=forecast_models,
    dgp_names=dgp_names,
    innovation_model_names=innovation_model_names,
    interval=interval,
    nominal_levels=nominal_levels,
)

save_table(
    main_results_df,
    dirs["tables"] / "main_calibration_results.csv",
)

main_results_df

,dgp,forecast_model,innovation_model,avg_coverage,avg_width,energy_score,crps,interval_score,ece,pit_deviation,coverage_1,width_1,coverage_2,width_2,coverage_3,width_3,nominal_coverage,coverage_error,abs_coverage_error
19,gaussian,RNN,diffusion,0.833333,3.241761,1.344740,0.663541,5.419067,0.058333,0.023333,0.850,3.326038,0.900,3.146078,0.750,3.253167,0.9,-6.666667e-02,6.666667e-02
17,gaussian,RNN,bootstrap,0.858333,3.092020,1.329566,0.659193,5.381033,0.072222,0.036667,0.825,3.108341,0.975,3.158400,0.775,3.009318,0.9,-4.166667e-02,4.166667e-02
16,gaussian,RNN,gaussian,0.825000,3.133914,1.332425,0.659789,5.328736,0.086111,0.025000,0.825,3.151952,0.900,3.068233,0.750,3.181556,0.9,-7.500000e-02,7.500000e-02
18,gaussian,RNN,student_t,0.800000,3.009000,1.347156,0.665151,5.483828,0.113889,0.036667,0.800,3.032123,0.875,2.935351,0.725,3.059526,0.9,-1.000000e-01,1.000000e-01
3,gaussian,VAR,diffusion,0.883333,3.699166,1.312506,0.655594,5.129345,0.013889,0.036667,0.875,4.062126,0.925,3.422134,0.850,3.613238,0.9,-1.666667e-02,1.666667e-02
2,gaussian,VAR,student_t,0.900000,3.570950,1.277481,0.639342,4.950210,0.016667,0.031667,0.875,3.882369,0.975,3.394392,0.850,3.436089,0.9,1.110223e-16,1.110223e-16
0,gaussian,VAR,gaussian,0.916667,3.670213,1.274063,0.637698,4.961294,0.019444,0.030000,0.875,3.935322,0.975,3.522003,0.900,3.553314,0.9,1.666667e-02,1.666667e-02
1,gaussian,VAR,bootstrap,0.883333,3.652746,1.286276,0.641768,5.039560,0.022222,0.033333,0.875,3.918968,0.975,3.524737,0.800,3.514531,0.9,-1.666667e-02,1.666667e-02
29,heteroskedastic,RNN,bootstrap,1.000000,6.454890,0.756082,0.374443,6.454890,0.183333,0.061667,1.000,7.059981,1.000,5.787584,1.000,6.517103,0.9,1.000000e-01,1.000000e-01
31,heteroskedastic,RNN,diffusion,1.000000,6.112297,0.781937,0.385960,6.112297,0.238889,0.073333,1.000,7.074286,1.000,5.814833,1.000,5.447772,0.9,1.000000e-01,1.000000e-01


In [4]:
display_cols = [
    "forecast_model",
    "dgp",
    "innovation_model",
    "avg_coverage",
    "abs_coverage_error",
    "avg_width",
    "ece",
    "pit_deviation",
    "crps",
    "energy_score",
    "interval_score",
]

compact_results_df = main_results_df[
    display_cols
].copy()

compact_results_df

,forecast_model,dgp,innovation_model,avg_coverage,abs_coverage_error,avg_width,ece,pit_deviation,crps,energy_score,interval_score
19,RNN,gaussian,diffusion,0.833333,6.666667e-02,3.241761,0.058333,0.023333,0.663541,1.344740,5.419067
17,RNN,gaussian,bootstrap,0.858333,4.166667e-02,3.092020,0.072222,0.036667,0.659193,1.329566,5.381033
16,RNN,gaussian,gaussian,0.825000,7.500000e-02,3.133914,0.086111,0.025000,0.659789,1.332425,5.328736
18,RNN,gaussian,student_t,0.800000,1.000000e-01,3.009000,0.113889,0.036667,0.665151,1.347156,5.483828
3,VAR,gaussian,diffusion,0.883333,1.666667e-02,3.699166,0.013889,0.036667,0.655594,1.312506,5.129345
2,VAR,gaussian,student_t,0.900000,1.110223e-16,3.570950,0.016667,0.031667,0.639342,1.277481,4.950210
0,VAR,gaussian,gaussian,0.916667,1.666667e-02,3.670213,0.019444,0.030000,0.637698,1.274063,4.961294
1,VAR,gaussian,bootstrap,0.883333,1.666667e-02,3.652746,0.022222,0.033333,0.641768,1.286276,5.039560
29,RNN,heteroskedastic,bootstrap,1.000000,1.000000e-01,6.454890,0.183333,0.061667,0.374443,0.756082,6.454890
31,RNN,heteroskedastic,diffusion,1.000000,1.000000e-01,6.112297,0.238889,0.073333,0.385960,0.781937,6.112297


In [5]:
from src.api.calibration import relative_improvement_table

relative_improvement_df = relative_improvement_table(
    main_results_df=main_results_df,
    forecast_models=forecast_models,
    dgp_names=dgp_names,
    innovation_model_names=innovation_model_names,
    baseline_model="gaussian",
)

save_table(
    relative_improvement_df,
    dirs["tables"] / "relative_improvement_vs_gaussian.csv",
)

relative_improvement_df

,forecast_model,dgp,innovation_model,ece_relative_improvement,pit_deviation_relative_improvement,crps_relative_improvement,energy_score_relative_improvement,interval_score_relative_improvement,abs_coverage_error_relative_improvement
0,VAR,gaussian,gaussian,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00
1,VAR,gaussian,bootstrap,-0.142857,-0.111111,-0.006381,-0.009586,-0.015775,1.332268e-14
2,VAR,gaussian,student_t,0.142857,-0.055556,-0.002577,-0.002683,0.002234,1.000000e+00
3,VAR,gaussian,diffusion,0.285714,-0.222222,-0.028063,-0.030174,-0.033872,6.661338e-15
4,VAR,student_t,gaussian,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00
5,VAR,student_t,bootstrap,-0.545455,-0.833333,-0.003676,-0.001737,-0.010939,-5.882353e-02
6,VAR,student_t,student_t,-0.272727,-0.666667,-0.002060,-0.001389,-0.025732,1.176471e-01
7,VAR,student_t,diffusion,-0.303030,-0.500000,-0.009157,-0.012120,-0.027001,1.176471e-01
8,VAR,mixture,gaussian,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00
9,VAR,mixture,bootstrap,-0.166667,-0.187500,0.011848,0.010236,-0.001846,-2.500000e-01


In [6]:
from src.api.calibration import headline_ranking_table

headline_ranking_df = headline_ranking_table(main_results_df=main_results_df,forecast_models=forecast_models,dgp_names=dgp_names,)
save_table(headline_ranking_df,dirs["tables"] / "headline_rankings.csv",)
headline_ranking_df

,forecast_model,dgp,best_ece_innovation,best_ece_value,best_pit_deviation_innovation,best_pit_deviation_value,best_crps_innovation,best_crps_value,best_energy_score_innovation,best_energy_score_value,best_interval_score_innovation,best_interval_score_value,best_abs_coverage_error_innovation,best_abs_coverage_error_value
0,VAR,gaussian,diffusion,0.013889,gaussian,0.030000,gaussian,0.637698,gaussian,1.274063,student_t,4.950210,student_t,1.110223e-16
1,VAR,student_t,gaussian,0.091667,gaussian,0.020000,gaussian,1.444045,gaussian,2.859699,gaussian,15.814952,student_t,1.250000e-01
2,VAR,mixture,student_t,0.025000,student_t,0.021667,bootstrap,0.996754,bootstrap,2.026307,gaussian,9.283644,student_t,3.333333e-02
3,VAR,heteroskedastic,bootstrap,0.236111,bootstrap,0.086667,bootstrap,0.431490,bootstrap,0.879109,student_t,6.403669,bootstrap,1.000000e-01
4,RNN,gaussian,diffusion,0.058333,diffusion,0.023333,bootstrap,0.659193,bootstrap,1.329566,gaussian,5.328736,bootstrap,4.166667e-02
5,RNN,student_t,gaussian,0.147222,gaussian,0.043333,gaussian,1.484820,gaussian,2.933800,gaussian,17.629813,gaussian,1.750000e-01
6,RNN,mixture,gaussian,0.050000,gaussian,0.025000,diffusion,1.040698,diffusion,2.110235,gaussian,9.942275,gaussian,5.000000e-02
7,RNN,heteroskedastic,bootstrap,0.183333,bootstrap,0.061667,bootstrap,0.374443,bootstrap,0.756082,student_t,5.735602,bootstrap,1.000000e-01


In [7]:
headline_ranking_df[["forecast_model",
                     "dgp",
                     "best_ece_innovation","best_ece_value",
                     "best_crps_innovation","best_crps_value",
                     "best_energy_score_innovation","best_energy_score_value",
                     "best_interval_score_innovation","best_interval_score_value",]]

,forecast_model,dgp,best_ece_innovation,best_ece_value,best_crps_innovation,best_crps_value,best_energy_score_innovation,best_energy_score_value,best_interval_score_innovation,best_interval_score_value
0,VAR,gaussian,diffusion,0.013889,gaussian,0.637698,gaussian,1.274063,student_t,4.950210
1,VAR,student_t,gaussian,0.091667,gaussian,1.444045,gaussian,2.859699,gaussian,15.814952
2,VAR,mixture,student_t,0.025000,bootstrap,0.996754,bootstrap,2.026307,gaussian,9.283644
3,VAR,heteroskedastic,bootstrap,0.236111,bootstrap,0.431490,bootstrap,0.879109,student_t,6.403669
4,RNN,gaussian,diffusion,0.058333,bootstrap,0.659193,bootstrap,1.329566,gaussian,5.328736
5,RNN,student_t,gaussian,0.147222,gaussian,1.484820,gaussian,2.933800,gaussian,17.629813
6,RNN,mixture,gaussian,0.050000,diffusion,1.040698,diffusion,2.110235,gaussian,9.942275
7,RNN,heteroskedastic,bootstrap,0.183333,bootstrap,0.374443,bootstrap,0.756082,student_t,5.735602


In [8]:
wins = []

metrics = [ "ece", "crps", "energy_score","interval_score",]

for metric in metrics:
    counts = ( headline_ranking_df[f"best_{metric}_innovation"].value_counts().to_dict())

    for innovation_model, n in counts.items():
        wins.append( { "metric": metric, "innovation_model": innovation_model,"wins": n,})

win_df = pd.DataFrame(wins)
win_df.sort_values(["metric", "wins"],ascending=[True, False],)

,metric,innovation_model,wins
4,crps,bootstrap,4
5,crps,gaussian,3
6,crps,diffusion,1
0,ece,gaussian,3
1,ece,diffusion,2
2,ece,bootstrap,2
3,ece,student_t,1
7,energy_score,bootstrap,4
8,energy_score,gaussian,3
9,energy_score,diffusion,1


In [9]:
win_pivot = win_df.pivot( index="innovation_model", columns="metric", values="wins",).fillna(0)
win_pivot

metric,crps,ece,energy_score,interval_score
innovation_model,,,,
bootstrap,4.0,2.0,4.0,0.0
diffusion,1.0,2.0,1.0,0.0
gaussian,3.0,3.0,3.0,5.0
student_t,0.0,1.0,0.0,3.0


Diffusion innovation models provide the greatest benefit when
innovation distributions exhibit complex non-Gaussian structure
such as mixtures and heteroskedasticity.

The largest gains appear in forecast distribution quality
(energy score and interval score), while improvements in
calibration metrics such as ECE are more context dependent.

Bootstrap:
better calibration

Diffusion:
better probabilistic forecasts